# Next-word prediction (perplexity) of the 20 models — original Google Colab notebook

This is the notebook that produced the next-word-prediction estimates used in Figure 2D and SI S15 (Methods, "Next-word prediction performance"; Supplementary Methods, "Next-word prediction task"). It was run on Google Colab (T4 GPU, high-RAM runtime), reading and writing files on Google Drive (`/content/drive/My Drive/...`), so it does not run as-is outside Colab. Code and outputs are kept exactly as they were run.

What it does:
1. Scrapes text from random Wikipedia pages in 16 languages (cells 3-4; regex tokens, i.e. words and punctuation marks). The scraped text was stored in `wikidict_all.pkl` on Google Drive and reloaded in cell 5; random pages cannot be re-scraped identically. Only the 12 Study I languages are used in the paper.
2. Defines the vocabulary as the 5,000 most frequent tokens per language and splits the text into 10-token windows; the target is the following token (out-of-vocabulary targets are mapped to `<UNK>`).
3. Extracts the last-layer contextual embedding of the last token of each window (`test_model_nwp`, cells 13 and 15-37; the first 50,000 tokens of each language).
4. Fits a ridge multi-class classifier (`RidgeClassifierCV`, alphas 1e-5 ... 1e4) with stratified 10-fold cross-validation and computes the held-out perplexity from the softmax of the decision function, for the real labels and for shuffled labels (random baseline) (`fit_ridge_classifier` and `test_model_ppx`, cells 38-40). Classes with fewer than 10 occurrences are grouped into `<UNK>` before fitting.

Notes for readers:
- The torch linear classifier defined in cell 12 (`fit_linear_classifier`) belongs to an earlier version and is not used for the reported results (`fit_classifier=False` by default); the reported perplexities come from the ridge classifier of cells 38-40.
- The printed values in the output of cell 40 are identical to the files `perplexity/PERPLEXITY_results_<model>.pkl` in this repository, which `perplexity/plot_perplexity.py` uses for Figure 2D.


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import pickle
import torch
import numpy as np
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, TensorDataset
from torch import nn, optim
import random
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import math
from collections import Counter
from os import path
from tqdm import tqdm
from transformers import XGLMTokenizer, XGLMForCausalLM, BertTokenizer, BertForMaskedLM, AutoTokenizer, AutoModelForMaskedLM, AutoModel, MT5EncoderModel, T5Tokenizer, XLMModel, XLMTokenizer, GPT2LMHeadModel, GPT2Tokenizer, DistilBertModel, DistilBertTokenizer, AutoModel, AutoTokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Fetching text from Wikipedia
As a first step, we fetch text from Wikipedia in 16 languages from random pages.

In [ ]:
# fetch text from Wikipedia in a given language
def fetch_wikipedia_text(lang='en', min_tokens=100000):
    tokens = []
    while len(tokens) < min_tokens:
        url = f"https://{lang}.wikipedia.org/wiki/Special:Random" # fetch random page
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser') # parse HTML content
        paragraphs = soup.find_all('p') # text from paragraphs
        text = ' '.join(p.text for p in paragraphs)
        tokens.extend(re.findall(r'\w+|[^\w\s]', text)) # splitting on whitespace and punctuation
        #print(f"Fetched {len(tokens)} tokens from: {url}")
        if len(tokens) >= min_tokens:
            break
    return tokens[:min_tokens]

In [ ]:
langs = ['ja', 'fr', 'ta', 'ca', 'es', 'tr', 'vi', 'en', 'mr', 'af', 'nl', 'no', 'fa', 'ro', 'lt', 'it']

wikidict = {}
for lang in langs:
  print(f"\n\nProcessing {lang}")
  wikidict[lang] = fetch_wikipedia_text(lang)

In [ ]:
filename = f"/content/drive/My Drive/alice/wikidict_all.pkl"
# with open(filename, 'wb') as file:
#   pickle.dump(wikidict, file)
with open(filename, 'rb') as file:
  wikidict = pickle.load(file)

# Define vocabulary for Next-word Prediction
We have data in 16 languages. We need to constrain the vocabulary for next-word prediction. We will consider the 5,000 most frequent words in each language, adding an extra token for out-of-vocabulary words.

In [ ]:
def get_vocabulary(token_list, n_items = 5000): # select top-n frequency items in each language
  counter = Counter(token_list)
  top_n = counter.most_common(n_items)
  top_n_strings = [item[0] for item in top_n] # remove counts
  return set(top_n_strings)

vocab_dict = {k : get_vocabulary(v) for k, v in wikidict.items()}

In [ ]:
# Dividing the wikipedia text stream into 10-word chunks.
def make_chunks(txt, size = 10):
  chunks  = []
  targets = []
  for i in range(len(txt)):
    try:
      chunk  = txt[i:i+size]
      target = txt[i+size]
      targets.append(target)
      chunks.append(chunk)
    except IndexError:
      pass
  return chunks, targets

## Extract embeddings from model
Functions to derive word-by-word embeddings from a given model (averaging over sub-word tokens).

In [ ]:
def tok_maker(a, sep, toker, cased = True):
    out = []
    for w in a:
        tok = toker.tokenize(w)
        tok[0] = re.sub(sep, "", tok[0])
        out.append(tok)
    return out

def get_word_embeddings(sentence, tokenizer, model, split_words = False):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if split_words: # for some models, we need to force word splitting
        input_ids = tokenizer.encode(sentence.split(), add_special_tokens=False, is_split_into_words=True, return_tensors="pt").to(device)
    else:
        input_ids = tokenizer.encode(sentence, add_special_tokens=False, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(input_ids, output_hidden_states=True)
    hidden_states = outputs.hidden_states
    layer_embeddings = hidden_states[-1][0].cpu().numpy()
    return layer_embeddings

def get_encoder_embeddings(sentence, tokenizer, model, split_words = False):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if split_words:
        inputs = tokenizer(sentence.split(), add_special_tokens=False, is_split_into_words=True, return_tensors="pt").to(device)
    else:
        inputs = tokenizer(sentence, add_special_tokens=False, return_tensors='pt').to(device)
    with torch.no_grad():
        encoder_outputs = model.get_encoder()(**inputs, output_hidden_states=True) # output of the encoder only
    hidden_states = encoder_outputs.hidden_states
    layer_embeddings = hidden_states[-1][0].cpu().numpy()
    return layer_embeddings

def get_embeddings_tokens(tokens, sep, tokenizer, model, cased=True, split_words = False, is_seq2seq = False):
    if is_seq2seq:
        layer_embs = get_encoder_embeddings(" ".join(tokens), tokenizer, model, split_words = split_words)
    else:
        layer_embs = get_word_embeddings(" ".join(tokens), tokenizer, model, split_words = split_words)
    #print(f"Layer embs shape = {layer_embs.shape}")
    toks = tok_maker(tokens, sep, tokenizer, cased)
    out_ = []
    theindex = 0
    for index, word in enumerate(toks):
        if len(word) == 1:
            emb = layer_embs[theindex]#; print(emb.shape)
            theindex += 1
            out_.append(emb)
        else:
            emb = layer_embs[theindex:theindex+len(word)].mean(axis=0)#; print(emb.shape)
            theindex += len(word)
            out_.append(emb)
    return out_

## Training classifier
Functions to train a linear classifier based on the models' embeddings and the dummy-coded next word. The code sets seeds for reproducibility. Training is done with linear NN (torch). Evaluation is performed with perplexity and 10-fold CV.

In [ ]:
def process_language(lang, sep, tokenizer, model, n_words = 50000, cased=True, split_words = False, is_seq2seq = False):
  # check that there are no weird tokens that the tokenizer ignores
  text = wikidict[lang]
  text_cleaned = []
  for tok in text:
    tknized = tokenizer.tokenize(tok)
    if len(tknized) == 0:
      text_cleaned.append("<UNK>")
    else:
      text_cleaned.append(tok)
  text_cleaned = text_cleaned[:n_words]
  # retrive vocabulary, make chunks
  voc  = vocab_dict[lang]
  chunks, classes = make_chunks(text_cleaned)
  # defining classes, adding <UNK> for OOV
  clean_classes = [cl_ if cl_ in voc else "<UNK>" for cl_ in classes]
  # extracting embeddings
  print(f"Extracting embeddings in {lang}")
  X = np.array([get_embeddings_tokens(chunk, sep, tokenizer, model, cased, split_words, is_seq2seq)[-1] for chunk in tqdm(chunks)])
  return X, clean_classes

def set_seed(seed_value=42): # seeting seeds for all possible RNG process just to be sure
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    random.seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class LinearClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super(LinearClassifier, self).__init__()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, x):
        return self.linear(x)

def fit_linear_classifier(X, y, k=10):
    set_seed()
    # choose device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    num_classes = len(np.unique(y_encoded))
    input_size = X.shape[1]
    # 10-fold CV
    kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    perplexities = []
    for train_index, test_index in tqdm(kf.split(X, y_encoded), total=k):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y_encoded[train_index], y_encoded[test_index]
        # np to torch
        X_train_torch = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_torch = torch.tensor(y_train, dtype=torch.long).to(device)
        X_test_torch = torch.tensor(X_test, dtype=torch.float32).to(device)
        y_test_torch = torch.tensor(y_test, dtype=torch.long).to(device)
        # dataset and loader
        train_dataset = TensorDataset(X_train_torch, y_train_torch)
        train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
        # init model
        model = LinearClassifier(input_size, num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        # train
        model.train()
        for epoch in range(10):  # 10 epochs
            for inputs, targets in train_loader:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()
        # eval
        model.eval()
        with torch.no_grad():
            outputs = model(X_test_torch)
            probabilities = torch.softmax(outputs, dim=1).cpu().numpy()
            loss = log_loss(y_test_torch.cpu().numpy(), probabilities, labels=range(num_classes))
            perplexity = math.exp(loss)
            perplexities.append(perplexity)
    average_perplexity = np.mean(perplexities)
    print(f"Average Perplexity: {average_perplexity}")
    return perplexities

In [ ]:
# putting everything together, saving outcome
def test_model_nwp(sep, modelname, tokenizer, model, n_words = 50000, considered_languages = wikidict.keys(), split_words = False, is_seq2seq = False, fit_classifier = False):
  results = {}
  for idx, lang in enumerate(considered_languages):
    dataname = f"/content/drive/My Drive/linear_readout/data_{modelname}_{lang}.pkl"
    filename = f"/content/drive/My Drive/linear_readout/perplexity_{modelname}_{lang}.pkl"
    filename_random = f"/content/drive/My Drive/linear_readout/perplexity_{modelname}_{lang}_random.pkl"
    if path.isfile(filename):
      print(f"{lang} already processed with {modelname}")
    else:
      print(f"Processing {lang} with {modelname} ({idx+1} out of {len(considered_languages)})")
      # get embeddings and class labels
      X, y = process_language(lang, sep="##", tokenizer=tokenizer, model=model, n_words = n_words, split_words = split_words, is_seq2seq = is_seq2seq)
      to_store = [X, y]
      with open(dataname, 'wb') as file:
        pickle.dump(to_store, file)
      if fit_classifier:
        # Original perplexity
        print("Fitting linear classifier (NN)")
        perplexities = fit_linear_classifier(X, y)
        # random perplexity
        print("Fitting linear classifier (random)")
        _, y_shuffled = shuffle(X, y, random_state=42)
        perplexities_random = fit_linear_classifier(X, y_shuffled)
        # storing
        results[lang] = {"result" : perplexities, "random" : perplexities_random}
        # Save to drive
        with open(filename, 'wb') as file:
          pickle.dump(perplexities, file)
        with open(filename_random, 'wb') as file:
          pickle.dump(perplexities_random, file)
  with open(f"/content/drive/My Drive/linear_readout/results_{modelname}.pkl", 'wb') as file:
    pickle.dump(results, file)
  return results

## Running analyses for specific models
Here the code runs the analyses for specific transformer-based models.

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
model = BertForMaskedLM.from_pretrained("bert-base-multilingual-cased")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
mbert_res = test_model_nwp("##", "mbert", tokenizer, model)

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-multilingual-cased")
model = DistilBertModel.from_pretrained("distilbert-base-multilingual-cased")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
distilmbert_res = test_model_nwp("##", "distilmbert", tokenizer, model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
model = AutoModelForMaskedLM.from_pretrained("xlm-roberta-base")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
xlmr_base_res = test_model_nwp("▁", "xlmr_base", tokenizer, model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")
model = AutoModelForMaskedLM.from_pretrained("xlm-roberta-large")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
xlmr_large_res = test_model_nwp("▁", "xlmr_large", tokenizer, model)

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("google/mt5-small")
model = MT5EncoderModel.from_pretrained("google/mt5-small")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
mt5_small_res = test_model_nwp("▁", "mt5_small", tokenizer, model)

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("google/mt5-base")
model = MT5EncoderModel.from_pretrained("google/mt5-base")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
mt5_base_res = test_model_nwp("▁", "mt5_base", tokenizer, model)

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("google/mt5-large")
model = MT5EncoderModel.from_pretrained("google/mt5-large")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
mt5_large_res = test_model_nwp("▁", "mt5_large", tokenizer, model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base")
model = AutoModel.from_pretrained("microsoft/mdeberta-v3-base")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
mdeberta_res = test_model_nwp("▁", "mdeberta", tokenizer, model, split_words = True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/xlm-align-base")
model = AutoModel.from_pretrained("microsoft/xlm-align-base")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
xlm_align_res = test_model_nwp("▁", "xlm_align", tokenizer, model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/infoxlm-base")
model = AutoModel.from_pretrained("microsoft/infoxlm-base")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
xlm_align_res = test_model_nwp("▁", "infoxlm_base", tokenizer, model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/infoxlm-large")
model = AutoModel.from_pretrained("microsoft/infoxlm-large")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
xlm_align_res = test_model_nwp("▁", "infoxlm_large", tokenizer, model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/Multilingual-MiniLM-L12-H384")
model = AutoModel.from_pretrained("microsoft/Multilingual-MiniLM-L12-H384")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
xlm_align_res = test_model_nwp("▁", "multiminilm", tokenizer, model)

In [ ]:
mgpt_langs   = ["af", "fa", "fr", "lt", "mr", "ro", "es", "ta", "tr", "vi"]
tokenizer = AutoTokenizer.from_pretrained("ai-forever/mGPT", add_prefix_space=True)
model = AutoModel.from_pretrained("ai-forever/mGPT")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
mgpt_res = test_model_nwp("Ġ", "mgpt", tokenizer, model, considered_languages = mgpt_langs, split_words = True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", add_prefix_space=True)
model = AutoModel.from_pretrained("facebook/nllb-200-distilled-600M")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
nllb200_d_600M = test_model_nwp("▁", "nllb200_distilled_600M", tokenizer, model, split_words = True, is_seq2seq = True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-1.3B", add_prefix_space=True)
model = AutoModel.from_pretrained("facebook/nllb-200-distilled-1.3B")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
nllb200_d_1b = test_model_nwp("▁", "nllb200_distilled_1B", tokenizer, model, split_words = True, is_seq2seq = True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-1.3B", add_prefix_space=True)
model = AutoModel.from_pretrained("facebook/nllb-200-1.3B")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
nllb200_d_1b = test_model_nwp("▁", "nllb200_1B", tokenizer, model, split_words = True, is_seq2seq = True)

## XGLM models
XGLM models only cover 8 languages (Catalan, Japanese, English, Spanish, Tamil, Turkish, French).

In [ ]:
xglm_langs = ["ca", "ja", "en", "es", "vi", "ta", "tr", "fr", "it"]
mgpt_langs = ["af", "fa", "fr", "lt", "mr", "ro", "es", "ta", "tr", "vi"]

In [ ]:
tokenizer = XGLMTokenizer.from_pretrained("facebook/xglm-564M")
model = XGLMForCausalLM.from_pretrained("facebook/xglm-564M")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
xglm_small_res = test_model_nwp("▁", "xglm_small", tokenizer, model, considered_languages = xglm_langs)

In [ ]:
tokenizer = XGLMTokenizer.from_pretrained("facebook/xglm-1.7B")
model = XGLMForCausalLM.from_pretrained("facebook/xglm-1.7B")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
xglm_med_res = test_model_nwp("▁", "xglm_med", tokenizer, model, considered_languages = xglm_langs)

In [ ]:
tokenizer = XGLMTokenizer.from_pretrained("facebook/xglm-2.9B")
model = XGLMForCausalLM.from_pretrained("facebook/xglm-2.9B")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
xglm_large_res = test_model_nwp("▁", "xglm_large", tokenizer, model, considered_languages = xglm_langs)

In [ ]:
tokenizer = XGLMTokenizer.from_pretrained("facebook/xglm-4.5B")
model = XGLMForCausalLM.from_pretrained("facebook/xglm-4.5B")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
xglm_xl_res = test_model_nwp("▁", "xglm_xl", tokenizer, model, considered_languages = xglm_langs)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("ai-forever/mGPT", add_prefix_space=True)
model = AutoModel.from_pretrained("ai-forever/mGPT")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
mgpt_res = test_model_nwp("Ġ", "mgpt", tokenizer, model, considered_languages = mgpt_langs, split_words = True)

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import log_loss

def group_minority_classes(y, threshold=10):
    unique, counts = np.unique(y, return_counts=True)
    minority_classes = unique[counts < threshold]
    y_grouped = np.where(np.isin(y, minority_classes), '<UNK>', y)
    return y_grouped

def fit_ridge_classifier(X, y, k_folds=10):
    # Grouping minority classes
    y_grouped = group_minority_classes(y)

    # Encoding labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_grouped)
    num_classes = len(label_encoder.classes_)

    # Stratified K-Fold setup
    kf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    perplexities = []

    # Specified alphas
    specified_alphas = np.array([0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000])

    for train_index, test_index in tqdm(kf.split(X, y_encoded), total = k_folds):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y_encoded[train_index], y_encoded[test_index]

        model = RidgeClassifierCV(alphas=specified_alphas, scoring='accuracy')
        model.fit(X_train, y_train)

        # Use the decision function to calculate probabilities
        decision_function = model.decision_function(X_test)
        exp_scores = np.exp(decision_function - np.max(decision_function, axis=1, keepdims=True))
        probabilities = exp_scores / exp_scores.sum(axis=1, keepdims=True)

        # Calculate log loss with the correct labels
        loss = log_loss(y_test, probabilities, labels=np.arange(num_classes))
        perplexity = np.exp(loss)
        perplexities.append(perplexity)

    average_perplexity = np.mean(perplexities)
    print(f"Average perplexity: {average_perplexity}")
    return average_perplexity


In [ ]:
# putting everything together, saving outcome
def test_model_ppx(modelname, considered_languages = wikidict.keys()):
  if path.isfile(f"/content/drive/My Drive/linear_readout/PERPLEXITY_results_{modelname}.pkl"):
    print(f"Model {modelname} already tested")
    with open(f"/content/drive/My Drive/linear_readout/PERPLEXITY_results_{modelname}.pkl", 'rb') as file:
      results = pickle.load(file)
    for k, v in results.items():
      print(f"{k}: {np.mean(v['result'])}")
  else:
    print(f"Testing model {modelname}")
    results = {}
    for idx, lang in enumerate(considered_languages):
      dataname = f"/content/drive/My Drive/linear_readout/data_{modelname}_{lang}.pkl"
      print("Attempting to open file:", dataname)
      with open(dataname, 'rb') as file:
        X, y = pickle.load(file)
      print(f"Loaded [|X| = {len(X)}, |y| ={len(y)}] examples for {lang}")
      # Original perplexity
      print(f"Fitting linear classifier for {lang}")
      perplexities = fit_ridge_classifier(X, y)
      # random perplexity
      print("Fitting linear classifier (random)")
      _, y_shuffled = shuffle(X, y, random_state=42)
      perplexities_random = fit_ridge_classifier(X, y_shuffled)
      # storing
      results[lang] = {"result" : perplexities, "random" : perplexities_random}
    with open(f"/content/drive/My Drive/linear_readout/PERPLEXITY_results_{modelname}.pkl", 'wb') as file:
      pickle.dump(results, file)
    return results

In [ ]:
for model in ["mbert", "distilmbert", "xlmr_base", "xlmr_large", "mt5_small", "mt5_base", "mt5_large", "xlm_align", "infoxlm_base", "infoxlm_large", "multiminilm", "nllb200_distilled_600M", "nllb200_distilled_1B", "nllb200_1B", "mdeberta"]:
  test_model_ppx(model)
  print(f"Done {model}")
for model in ["xglm_small", "xglm_med", "xglm_large", "xglm_xl"]:
  test_model_ppx(model, considered_languages = xglm_langs)
  print(f"Done {model}")
for model in ["mgpt"]:
  test_model_ppx(model, considered_languages = mgpt_langs)
  print(f"Done {model}")

Model mbert already tested
ja: 53.35439827990107
fr: 324.2180591985986
ta: 141.41059821756372
ca: 306.89011105136234
es: 298.5440834916217
tr: 277.86440188232774
vi: 514.9700784323161
en: 381.91737326994775
mr: 153.08888597450897
af: 327.81344242475103
nl: 294.24545702874406
no: 290.27080379219296
fa: 411.3442540147448
ro: 174.97786062281298
lt: 232.7335555790728
it: 384.20563745698024
Done mbert
Model distilmbert already tested
ja: 53.463338543679036
fr: 327.8103893490984
ta: 140.80960703638885
ca: 311.6956549252827
es: 304.94440361920664
tr: 283.34557444099545
vi: 531.178859129074
en: 390.03429051884
mr: 153.32627173691648
af: 333.5021627211261
nl: 298.8724258497426
no: 297.12908609257147
fa: 415.8255247244553
ro: 177.64645802312228
lt: 240.0419117137431
it: 389.2317337675651
Done distilmbert
Model xlmr_base already tested
ja: 55.883799761606724
fr: 332.66919102475345
ta: 145.16353502643784
ca: 316.75966839532856
es: 307.332776689646
tr: 285.9206530795041
vi: 564.3561120333445
en: 39

100%|██████████| 10/10 [01:50<00:00, 11.03s/it]


Average perplexity: 60.066635483880304
Fitting linear classifier (random)


100%|██████████| 10/10 [01:47<00:00, 10.72s/it]


Average perplexity: 103.51007069005736
Attempting to open file: /content/drive/My Drive/linear_readout/data_mdeberta_fr.pkl
Loaded [|X| = 49990, |y| =49990] examples for fr
Fitting linear classifier for fr


...

100%|██████████| 10/10 [02:49<00:00, 16.91s/it]


Average perplexity: 421.58582580771656
Fitting linear classifier (random)


100%|██████████| 10/10 [02:46<00:00, 16.68s/it]

Average perplexity: 458.23059690320287
Done mdeberta


NameError: name 'xglm_langs' is not defined